# PV feed-in and grid draw in example district scenarios

This notebook analyzes a previously assembled **multi-scenario PEXL project** with hourly timestep data.

Create the project first with `howto_create_project_from_different_timeseries_files.ipynb`. This keeps file collection and project assembly separate from the actual scenario analysis.


## Load the prepared project

The saved project already contains the scenarios and their timestep data, so the source Excel files do not need to be reopened here.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pickle
import pandas as pd
import plotly.io as pio

pio.renderers.default = "notebook"
print(pio.renderers.default)


In [ ]:


project_path = Path(
    r"C:\Users\schneids\Nextcloud\EE\6_Daten\Quartiere"
    r"\1220_Wien_Aspern_Seestadt\Unterlagen PV Überschüsse"
    r"\PEexcel Auswertung\aspern_pvcheck_project.pkl"
)

with project_path.open("rb") as file:
    project = pickle.load(file)

project


In [ ]:
import pexl.plot.plotly
fig = pexl.plot.plotly.chart(project, "heat_balance")
fig.show(renderer="notebook")


In [ ]:

fig = pexl.plot.plotly.chart(project, "primary_energy_balance")
fig.show(renderer="notebook")

## Annual PV indicators

The PEXL project view is used to select only the annual indicators relevant to PV utilization and grid feed-in. This provides a compact comparison across all scenarios without manually accessing each scenario object.

In [ ]:
# if you dont know the names, you can use the pexl glossary to find and specify them

import pexl 
pexl.glossary.GFA_total


In [ ]:


pv_metrics = project.out.select(
    pexl.glossary.PV_own_consumption,
    pexl.glossary.PV_own_consumption_flex,
    pexl.glossary.PV_own_consumption,
    pexl.glossary.EUI_self_sufficiency,
    pexl.glossary.PV_peak_grid_feedin,
    pexl.glossary.PV_peak_grid_feedin_date,
)
pv_metrics


## Hourly grid interaction

The exported timestep values are specific energy quantities in Wh/m² per hourly timestep. Multiplication by the scenario gross floor area (`GFA_total`) and division by 1000 converts them to kWh per hour, numerically equivalent to the mean **kW** during that hour.

Two comparable datasets are prepared for every scenario:

- **Grid draw**: electricity drawn from the grid.
- **PV feed-in**: PV electricity exported to the grid.

The same data are also arranged as a nested mapping for direct multi-scenario plotting with `pexl.plot.duration_curve()`.

In [ ]:
grid_draw = {}
pv_feedin = {}
flow_data = {}

for scenario in project:
    df = scenario.timeseries
    factor = scenario.v.NFA_total / 1000

    grid = df["E_grid"] * factor
    feedin = df["PV_to_Egrid"] * factor

    grid_draw[scenario.column_name] = grid
    pv_feedin[scenario.column_name] = feedin

    flow_data[scenario.column_name.split(" _ ")[1]] = {
        "Netzbezug": grid,
        "Netzeinspeisung": feedin,
    }

grid_draw = pd.DataFrame(grid_draw)
pv_feedin = pd.DataFrame(pv_feedin)

In [ ]:
SUPERBLOCKS = {
    "Superblock F5+F6+F7": ["F5", "F6", "F7"],
    "Superblock F9B+F10+F12+F13": ["F9B", "F10", "F12", "F13"],
}

for block_name, members in SUPERBLOCKS.items():
    flow_data[block_name] = {
        flow_name: pd.concat(
            [flow_data[member][flow_name] for member in members],
            axis=1,
        ).sum(axis=1)
        for flow_name in ["Netzbezug", "Netzeinspeisung"]
    }

## Hourly and seasonal patterns

Heatmaps show **when** high grid interaction occurs during the year. Grid draw and PV feed-in are plotted separately so that each can use a suitable sequential color scale.

In [ ]:
from matplotlib import pyplot as plt


scenario_name = "F7"

pexl.plot.heatmap(
    {
        "Netzbezug": -flow_data[scenario_name]["Netzbezug"],
        "Netzeinspeisung": flow_data[scenario_name]["Netzeinspeisung"],
    },
    unit="kW",
    layout="vertical",
    center=0,
    cmap="PiYG",
    language="de"
)
plt.suptitle(scenario_name)

In [ ]:


scenario_name = "F7"

pexl.plot.heatmap(
    {
        "Bilanz": (
            flow_data[scenario_name]["Netzeinspeisung"]
            - flow_data[scenario_name]["Netzbezug"]
        ),
    },
    unit="kW",
    layout="vertical",
    center=0,
    cmap=pexl.plot.styles.BALANCE_CMAP,
    language="de",
)

plt.suptitle(scenario_name)

In [ ]:
for scenario_name, flows in flow_data.items():

    pexl.plot.heatmap(
        {
            "Bilanz": (
                flows["Netzeinspeisung"]
                - flows["Netzbezug"]
            ),
        },
        unit="kW",
        layout="vertical",
        center=0,
        cmap=pexl.plot.styles.BALANCE_CMAP,
        language="de",
    )

    plt.suptitle(scenario_name)
    plt.show()

## Duration curves: magnitude and persistence of grid strain

The annual duration curves sort all hourly values from highest to lowest. They therefore show both:

- the **magnitude** of peak grid draw and PV feed-in,
- and the **number of hours** for which high grid loads persist.

Each row represents one district scenario. The second panel zooms into the first `zoom_hours` hours so that the most grid-relevant peak events can be compared more closely.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import pexl

output_dir = Path(
    r"C:\Users\schneids\Nextcloud\EE\6_Daten\Quartiere\1220_Wien_Aspern_Seestadt\Unterlagen PV Überschüsse\PEexcel Auswertung"
)
# Detailed view of the first hours of each duration curve.
zoom_hours = 7*24
output_dir.mkdir(parents=True, exist_ok=True)

flow_colors = {
    "Netzbezug": "firebrick",
    "Netzeinspeisung": "limegreen",
}

# do_save = input(f"save duration_curve under {output_dir.name}? (y)") == "y"

# for scenario_name, flows in flow_data.items():
#     fig, axes = pexl.plot.duration_curve(
#         flows,
#         unit="kW",
#         peak_hours=zoom_hours,
#         colors=flow_colors,
#     )

#     fig.suptitle(scenario_name, y=1.01)
    
#     short_name = scenario_name.split(" _ ")[-1]
#     filename = f"Load curve _ {short_name}.png"
#     if do_save:
#         fig.savefig(output_dir / filename, dpi=300, bbox_inches="tight")

#     plt.show()

In [ ]:
from pexl.plot import seasonal_duration_curves
# do_save = input(f"save seasonal_duration_curves under {output_dir.name}? (y)") == "y"
for scenario_name, flows in flow_data.items():
    fig, axes = pexl.plot.seasonal_duration_curves(
        flows,
        unit="kW",
        colors=flow_colors,
    )

    fig.suptitle(scenario_name, y=1.01)
    
    short_name = scenario_name.split(" _ ")[-1]
    filename = f"Load curve _ {short_name}.png"
    # if do_save: 
    #     fig.savefig(output_dir / filename, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
from pexl.plot import duration_curve_interactive
fig = pexl.plot.duration_curve_interactive(
    flows,
    ylabel="Leistung [kW]",
    colors=flow_colors,
)
fig.show(renderer="notebook")

## Interpretation

The heatmaps and duration curves complement the annual indicators:

- **Peak magnitude** indicates the maximum local power that may affect connection or grid-capacity requirements.
- **Duration at high power** indicates whether these peaks are isolated events or persistent operating conditions.
- **PV feed-in versus grid draw** shows whether export peaks are of a similar, smaller, or larger magnitude than the district's expected import peaks.
- **Seasonal and hourly timing** helps distinguish PV-driven daytime/summer export stress from demand-driven grid draw.

The analysis is intentionally based on hourly district energy balances. It characterizes expected grid-facing power profiles, but does not replace a detailed electrical network calculation with voltage, line-loading, transformer, or power-flow constraints.

In [ ]:

from pexl.plot import typical_day_balance
DAYS =["2018-06-21"]

# do_save = input(f"save typical_day_balance?") =="y"

# for DAY in DAYS:
#     for name, flow in flow_data.items():
#         title = f"{name} - {DAY}"
#         fig = typical_day_balance(
#             flow["Netzeinspeisung"],
#             flow["Netzbezug"],
#             day=DAY,
#             title=title,
#             ylabel="Leistung [kW]",
#         )
#         fig.show(renderer="notebook")
#         #fig.write_html(output_dir /  f"typicaldays/{title}.html")
#         # if do_save:
#         #     fig.write_image(
#         #         output_dir / f"typicaldays/{title}.png",
#         #         width=800,
#         #         height=300,
#         #         scale=2,
#         #     )

In [ ]:
from pexl.plot import seasonal_hourly_boxplot

# do_save = input("save seasonal_hourly_boxplot?") =="y"
for name, flow in flow_data.items():
    fig = seasonal_hourly_boxplot(
        flow["Netzeinspeisung"]-flow["Netzbezug"],
        ylabel="Leistungssaldo [kW]",
        title=name,
        shade_sign=True,
    )
    fig.show(renderer="notebook")
    # if do_save:
    #     fig.write_image(
    #         output_dir / f"seasonal_balances/{name}.png",
    #         width=1200,
    #         height=400,
    #         scale=1,
    #     )

In [ ]:
!jupyter nbconvert --to html howto_analyze_gridload.ipynb

In [ ]:
import nbformat

nb = nbformat.read("howto_analyze_gridload.ipynb", as_version=4)

for i, cell in enumerate(nb.cells):
    for output in cell.get("outputs", []):
        if output.output_type in {"display_data", "execute_result"}:
            mime = list(output.get("data", {}).keys())

            if "application/vnd.plotly.v1+json" in mime:
                print(i, mime)